In [6]:
# Cell 1 — Setup
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

from scripts.shared.db_utils import db_connect
conn = db_connect()
print('connected')

connected


## Step 1 items

Items addressed in this notebook:

1. **Paint path** — is the current LMR path already slice-synced, or is it notch-based?
2. **LMR floor** — what is the actual effective floor, and what does the endpoint return below it?
3. **Zero-width slices** — does `fromyear=toyear` (e.g. 875–875 CE) produce a valid result?
4. **BCE / Qin** — is a BCE polity queryable in the current app at all?
5. **Period vs. slice discrepancy** — labelling artifact or real defect?
6. **Extensive polity LMR variation** — does the LMR field vary visibly within a large polity at a single slice span? (Abbasid, Tibetan Empire)

In [4]:
# Cell 2 — LMR table structure
sql = """
SELECT
    COUNT(*)                          AS n_cells,
    MIN(lat)                          AS lat_min,
    MAX(lat)                          AS lat_max,
    MIN(lon)                          AS lon_min,
    MAX(lon)                          AS lon_max,
    MAX(array_length(prate, 1))       AS prate_len,
    MAX(array_length(air,   1))       AS air_len,
    MAX(array_length(pdsi,  1))       AS pdsi_len
FROM temporal.lmr_climate
"""
r = pd.read_sql(sql, conn).iloc[0]
for col, val in r.items():
    print(f'{col:20s}: {val}')

/var/folders/f9/r5mr431d23zcpktsjz_xjxt40000gn/T/ipykernel_76909/183523340.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(sql, conn).iloc[0]


### Item 1 — Paint path (code inspection result)

After the WO1 LMR-repaint fix, `applySlice()` writes `s.fromyear` / `s.toyear` to the
Band T inputs. `_repaintChoropleth()` reads those via `_activeBandTYears()` and calls
`applyLMRChoropleth(key, fromYr, toYr)`. That function fetches
`/api/lmr/values?var=X&from_year=Y&to_year=Z`.

That endpoint (`routes.py` line 2840) runs:
```sql
SELECT AVG(v) FROM unnest(var[actual_from : to_year]) AS v
```
directly on `temporal.lmr_climate` annual arrays — **no notch table involved**.

**Finding: the LMR paint path is already slice-synced.** The fix made in the previous session
already wires `applySlice()` → Band T inputs → `/api/lmr/values` with per-slice spans.
Cell 3 below verifies this produces different values for different N Song slice spans.

In [7]:
# Cell 3 — Spot-check: do different N Song slice spans give different global LMR means?
# N Song has three distinct states: spans 961, 970-979, 980+.
# We compute the global mean of prate (precip anomaly) over each span.
# If values differ → slice-synced paint will show different pictures.
FLOOR = 700  # hardcoded in /api/lmr/values

spans = [
    ('N Song slice 1 (961–961)',   961, 961),
    ('N Song slice 3 (970–979)',   970, 979),
    ('N Song slice 4 (980–989)',   980, 989),
    ('N Song slice 5 (990–1017)', 990, 1017),
]

rows = []
for label, y1, y2 in spans:
    sql = f"""
        SELECT
            AVG((SELECT AVG(v) FROM unnest(prate[{y1}:{y2}]) AS v))  AS prate_global_mean,
            AVG((SELECT AVG(v) FROM unnest(air  [{y1}:{y2}]) AS v))  AS air_global_mean,
            COUNT(*)                                                  AS n_cells
        FROM temporal.lmr_climate
    """
    r = pd.read_sql(sql, conn).iloc[0]
    rows.append({'span': label, 'prate_mean': round(r.prate_global_mean, 6),
                 'air_mean': round(r.air_global_mean, 6), 'n_cells': int(r.n_cells)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print('prate range across spans:', round(df.prate_mean.max() - df.prate_mean.min(), 6))
print('air range across spans:  ', round(df.air_mean.max()   - df.air_mean.min(), 6))

                     span  prate_mean  air_mean  n_cells
 N Song slice 1 (961–961)        -0.0 -0.190416    16380
 N Song slice 3 (970–979)        -0.0 -0.169540    16380
 N Song slice 4 (980–989)        -0.0 -0.164772    16380
N Song slice 5 (990–1017)        -0.0 -0.178278    16380

prate range across spans: 0.0
air range across spans:   0.025644


In [9]:
# Cell 4 — LMR floor behaviour
cases = [
    ('below floor (600–699)',            600, 699,  700),
    ('Tibetan Empire (623–840)',          623, 840,  700),
    ('floor exact (700–700)',             700, 700,  700),
    ('zero-width above floor (875–875)', 875, 875,  875),
    ('N Song first slice (961–961)',      961, 961,  961),
]

for label, fy, ty, actual_from in cases:
    print(f'--- {label} ---', flush=True)
    if actual_from > ty:
        print(f'  EMPTY — below floor (actual_from={actual_from} > to_year={ty})')
        continue
    try:
        sql = f"""
            SELECT COUNT(*) AS n_cells,
                   AVG((SELECT AVG(v) FROM unnest(prate[{actual_from}:{ty}]) AS v)) AS mean_prate
            FROM temporal.lmr_climate
        """
        r = pd.read_sql(sql, conn).iloc[0]
        prate_val = r.mean_prate
        print(f'  n_cells={int(r.n_cells)}, mean_prate raw={prate_val!r}, type={type(prate_val).__name__}')
        if prate_val is not None:
            print(f'  mean_prate rounded={round(float(prate_val), 6)}')
    except Exception as e:
        print(f'  ERROR: {type(e).__name__}: {e}')

  n_cells=16380, mean_prate raw=np.float64(-2.946457665487145e-07), type=float64
  mean_prate rounded=-0.0


In [10]:
# Cell 5 — BCE / Qin (Item 4)
# Does a BCE polity exist in gaz.clio_polities and what are its spans?
# Does the current /api/lmr/values logic handle negative years?

sql = """
    SELECT name, fromyear, toyear, id
    FROM gaz.clio_polities
    WHERE name ILIKE '%qin%'
      AND NOT is_component
    ORDER BY fromyear
"""
qin = pd.read_sql(sql, conn)
print('Qin polity slices in gaz.clio_polities:')
print(qin.to_string(index=False))
print()

# Simulate what /api/lmr/values does for a BCE span.
# actual_from = max(from_year, 700); if actual_from > to_year → empty dict.
for _, row in qin.iterrows():
    fy, ty = int(row.fromyear), int(row.toyear)
    actual_from = max(fy, 700)
    result = 'EMPTY (actual_from=%d > to_year=%d)' % (actual_from, ty) if actual_from > ty else 'QUERYABLE (actual_from=%d)' % actual_from
    print(f'  {row["name"]} {fy}–{ty}: {result}')

Qin polity slices in gaz.clio_polities:
        name  fromyear  toyear    id
         Qin      -750    -451   142
         Qin      -450    -405   359
         Qin      -404    -367   372
         Qin      -366    -351   414
         Qin      -350    -338   453
         Qin      -337    -327   474
         Qin      -326    -316   501
         Qin      -315    -278   537
         Qin      -277    -231   656
         Qin      -230    -226   779
         Qin      -225    -224   790
         Qin      -223    -223   829
         Qin      -222    -219   833
 Qin Dynasty      -218    -213   844
 Qin Dynasty      -212    -211   876
 Qin Dynasty      -210    -209   885
  Former Qin       353     370  1762
  Former Qin       371     372  1790
  Former Qin       373     377  1801
  Former Qin       378     382  1810
  Former Qin       383     386  1817
   Later Qin       387     393  1839
  Former Qin       387     389  1828
 Western Qin       387     393  1824
  Former Qin       390     391  185

In [11]:
# Cell 6 — Period vs. slice discrepancy (Item 5)
# The sandbox showed Northern Song period "1000–1100 CE" but active slice "990–1017 CE".
# The /api/polity/period endpoint returns period dates. Let's see what it does.
# Also confirm: what are the actual N Song slice bounds in gaz.clio_polities?

sql = """
    SELECT name, fromyear, toyear, id,
           ROW_NUMBER() OVER (PARTITION BY name ORDER BY fromyear) AS slice_n,
           COUNT(*) OVER (PARTITION BY name)                       AS total_slices
    FROM gaz.clio_polities
    WHERE name = 'Northern Song'
      AND NOT is_component
    ORDER BY fromyear
"""
nsong = pd.read_sql(sql, conn)
print('N Song slices in gaz.clio_polities:')
print(nsong.to_string(index=False))
print()
print('Full polity lifespan:', nsong.fromyear.min(), '–', nsong.toyear.max())

N Song slices in gaz.clio_polities:
         name  fromyear  toyear   id  slice_n  total_slices
Northern Song       961     961 4352        1             6
Northern Song       962     969 4367        2             6
Northern Song       970     979 4401        3             6
Northern Song       980     989 4454        4             6
Northern Song       990    1017 4481        5             6
Northern Song      1018    1027 4683        6             6

Full polity lifespan: 961 – 1027


In [12]:
# Cell 7 — Abbasid Caliphate slices and peak extent (Item 6 setup)
# Find Abbasid slices; identify the peak (largest basin count).

sql = """
    SELECT p.name, p.fromyear, p.toyear, p.id,
           COUNT(b.hybas_id) AS n_basins
    FROM gaz.clio_polities p
    LEFT JOIN public.basin06 b
           ON ST_Within(ST_Centroid(b.geom), p.geom)
    WHERE p.name ILIKE '%abbasid%'
      AND NOT p.is_component
    GROUP BY p.name, p.fromyear, p.toyear, p.id
    ORDER BY p.fromyear
"""
abbasid = pd.read_sql(sql, conn)
print('Abbasid slices (centroid-in basin count):')
print(abbasid.to_string(index=False))
print()
# Identify the LMR-reachable slices (fromyear or toyear >= 700)
abbasid['lmr_actual_from'] = abbasid['fromyear'].apply(lambda y: max(int(y), 700))
abbasid['lmr_ok'] = abbasid.apply(lambda r: int(r.lmr_actual_from) <= int(r.toyear), axis=1)
print('LMR-reachable slices (actual_from <= toyear):')
print(abbasid[['name','fromyear','toyear','n_basins','lmr_actual_from','lmr_ok']].to_string(index=False))

Abbasid slices (centroid-in basin count):
                           name  fromyear  toyear   id  n_basins
              Abbasid Caliphate       750     750 3142       371
              Abbasid Caliphate       751     754 3178       931
              Abbasid Caliphate       755     756 3189       972
              Abbasid Caliphate       757     762 3218       978
              Abbasid Caliphate       763     767 3236       999
              Abbasid Caliphate       768     777 3259      1002
              Abbasid Caliphate       778     782 3292      1005
              Abbasid Caliphate       783     799 3316      1002
              Abbasid Caliphate       800     813 3384       978
              Abbasid Caliphate       814     824 3423       977
              Abbasid Caliphate       825     829 3443       962
              Abbasid Caliphate       830     839 3459       962
              Abbasid Caliphate       840     849 3491       977
              Abbasid Caliphate       850     85

In [13]:
# Cell 8 — Abbasid LMR spread at peak slice (the extent hero-shot test)
peak = abbasid.loc[abbasid.n_basins.idxmax()]
peak_id   = int(peak['id'])
peak_fy   = int(peak['fromyear'])
peak_ty   = int(peak['toyear'])
actual_fy = max(peak_fy, 700)
print(f'Peak slice: id={peak_id}, {peak_fy}–{peak_ty} CE, n_basins={int(peak.n_basins)}')
print(f'LMR span used: {actual_fy}–{peak_ty} CE')

sql = f"""
    SELECT
        COUNT(*)                                                           AS n_cells,
        PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY mean_prate)          AS p10_prate,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY mean_prate)          AS p50_prate,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY mean_prate)          AS p90_prate,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY mean_prate)
          - PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY mean_prate)      AS spread_prate,
        PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY mean_air)            AS p10_air,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY mean_air)            AS p50_air,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY mean_air)            AS p90_air,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY mean_air)
          - PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY mean_air)        AS spread_air
    FROM (
        SELECT l.lat, l.lon,
               (SELECT AVG(v) FROM unnest(l.prate[{actual_fy}:{peak_ty}]) AS v) AS mean_prate,
               (SELECT AVG(v) FROM unnest(l.air  [{actual_fy}:{peak_ty}]) AS v) AS mean_air
        FROM temporal.lmr_climate l
        JOIN gaz.clio_polities    p ON p.id = {peak_id}
        WHERE ST_Within(l.geom, p.geom)
    ) sub
"""
r = pd.read_sql(sql, conn).iloc[0]
print()
for col, val in r.items():
    print(f'{col:15s}: {round(float(val), 6) if val is not None else None}')

Peak slice: id=3292, 778–782 CE, n_basins=1005
LMR span used: 778–782 CE

n_cells        : 197.0
p10_prate      : -0.0
p50_prate      : -0.0
p90_prate      : 0.0
spread_prate   : 1e-06
p10_air        : -0.106994
p50_air        : -0.056875
p90_air        : 0.011487
spread_air     : 0.118481


In [14]:
# Cell 8b — Verify: are the 197 cells actually inside Abbasid territory?
# Show geographic bounds and a sample of individual cell values to confirm
# the spatial join is working and the values are genuinely non-uniform.
sql = f"""
    SELECT
        l.lat,
        CASE WHEN l.lon > 180 THEN l.lon - 360 ELSE l.lon END AS lon,
        (SELECT AVG(v) FROM unnest(l.air  [778:782]) AS v) AS mean_air,
        (SELECT AVG(v) FROM unnest(l.prate[778:782]) AS v) AS mean_prate
    FROM temporal.lmr_climate l
    JOIN gaz.clio_polities p ON p.id = 3292
    WHERE ST_Within(l.geom, p.geom)
    ORDER BY l.lat DESC, l.lon
"""
cells = pd.read_sql(sql, conn)
print(f'Matched cells : {len(cells)}')
print(f'Lat range     : {cells.lat.min():.0f} – {cells.lat.max():.0f} °N')
print(f'Lon range     : {cells.lon.min():.0f} – {cells.lon.max():.0f} °E')
print(f'air  min/max  : {cells.mean_air.min():.6f} / {cells.mean_air.max():.6f}  K')
print(f'prate min/max : {cells.mean_prate.min():.6e} / {cells.mean_prate.max():.6e}  mm/day')
print()
print('Sample — northernmost 10 cells:')
print(cells.head(10)[['lat','lon','mean_air','mean_prate']].to_string(index=False))
print()
print('Sample — southernmost 10 cells:')
print(cells.tail(10)[['lat','lon','mean_air','mean_prate']].to_string(index=False))

Matched cells : 197
Lat range     : 14 – 42 °N
Lon range     : 2 – 72 °E
air  min/max  : -0.174954 / 0.039887  K
prate min/max : -7.100027e-07 / 3.936201e-07  mm/day

Sample — northernmost 10 cells:
 lat  lon  mean_air    mean_prate
42.0  2.0 -0.174954 -9.197837e-08
42.0 44.0 -0.114990 -3.005214e-07
42.0 46.0 -0.099085 -2.653483e-07
42.0 58.0 -0.166915 -1.048073e-07
42.0 60.0 -0.162555 -6.807160e-08
42.0 68.0 -0.123809 -1.875540e-07
42.0 70.0 -0.136361 -2.901969e-07
40.0 40.0 -0.157893 -3.275084e-07
40.0 42.0 -0.113056 -2.630859e-07
40.0 44.0 -0.112079 -2.190175e-07

Sample — southernmost 10 cells:
 lat  lon  mean_air    mean_prate
18.0 54.0 -0.043471  1.305705e-07
18.0 56.0 -0.027132 -9.150914e-08
16.0 44.0 -0.076663  2.627095e-07
16.0 46.0 -0.095232  5.656265e-08
16.0 48.0 -0.076383  2.584133e-07
16.0 50.0 -0.050146  1.969145e-07
16.0 52.0 -0.027366  1.752026e-08
14.0 44.0 -0.045268 -1.933511e-07
14.0 46.0 -0.038314 -7.303757e-08
14.0 48.0 -0.027901  6.824876e-08


In [15]:
# Cell 9 — Tibetan Empire: partial floor overlap
# Tibetan Empire spans 623–840 CE; LMR floor is 700 CE.
# How many slices are entirely below floor, how many straddle, how many are above?

sql = """
    SELECT name, fromyear, toyear, id
    FROM gaz.clio_polities
    WHERE name ILIKE '%tibetan empire%'
      AND NOT is_component
    ORDER BY fromyear
"""
tibet = pd.read_sql(sql, conn)
tibet['actual_from'] = tibet['fromyear'].apply(lambda y: max(int(y), 700))
tibet['lmr_status'] = tibet.apply(
    lambda r: 'EMPTY (below floor)' if int(r.actual_from) > int(r.toyear)
              else ('straddles floor' if int(r.fromyear) < 700
              else 'full LMR coverage'),
    axis=1
)
print('Tibetan Empire slices vs LMR 700 CE floor:')
print(tibet[['name','fromyear','toyear','actual_from','lmr_status']].to_string(index=False))

Tibetan Empire slices vs LMR 700 CE floor:
          name  fromyear  toyear  actual_from          lmr_status
Tibetan Empire       623     626          700 EMPTY (below floor)
Tibetan Empire       627     627          700 EMPTY (below floor)
Tibetan Empire       628     633          700 EMPTY (below floor)
Tibetan Empire       634     637          700 EMPTY (below floor)
Tibetan Empire       638     640          700 EMPTY (below floor)
Tibetan Empire       641     655          700 EMPTY (below floor)
Tibetan Empire       656     660          700 EMPTY (below floor)
Tibetan Empire       661     665          700 EMPTY (below floor)
Tibetan Empire       666     673          700 EMPTY (below floor)
Tibetan Empire       674     681          700 EMPTY (below floor)
Tibetan Empire       682     691          700 EMPTY (below floor)
Tibetan Empire       692     704          700     straddles floor
Tibetan Empire       705     717          705   full LMR coverage
Tibetan Empire       718     731 

In [16]:
# Cell 10 — Tang Dynasty: same floor check + LMR spread test
# Tang (618–907 CE) starts just below the floor (618 < 700).
# Check slices; run the spread query for a peak slice.

sql = """
    SELECT p.name, p.fromyear, p.toyear, p.id,
           COUNT(b.hybas_id) AS n_basins
    FROM gaz.clio_polities p
    LEFT JOIN public.basin06 b ON ST_Within(ST_Centroid(b.geom), p.geom)
    WHERE p.name = 'Tang Dynasty'
      AND NOT p.is_component
    GROUP BY p.name, p.fromyear, p.toyear, p.id
    ORDER BY p.fromyear
"""
tang = pd.read_sql(sql, conn)
tang['actual_from'] = tang['fromyear'].apply(lambda y: max(int(y), 700))
tang['lmr_ok'] = tang.apply(lambda r: int(r.actual_from) <= int(r.toyear), axis=1)
print('Tang Dynasty slices:')
print(tang[['name','fromyear','toyear','n_basins','actual_from','lmr_ok']].to_string(index=False))

# LMR spread for peak LMR-reachable slice
tang_lmr = tang[tang.lmr_ok]
if len(tang_lmr) == 0:
    print('\nNo Tang slices with LMR coverage.')
else:
    peak_t = tang_lmr.loc[tang_lmr.n_basins.idxmax()]
    pid = int(peak_t['id']); fy = int(peak_t['actual_from']); ty = int(peak_t['toyear'])
    print(f'\nRunning LMR spread for Tang peak: id={pid}, LMR span {fy}–{ty} CE, n_basins={int(peak_t.n_basins)}')
    sql2 = f"""
        SELECT COUNT(*) AS n_cells,
               PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY mp) - PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY mp) AS spread_prate,
               PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY ma) - PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY ma) AS spread_air
        FROM (
            SELECT (SELECT AVG(v) FROM unnest(l.prate[{fy}:{ty}]) AS v) AS mp,
                   (SELECT AVG(v) FROM unnest(l.air  [{fy}:{ty}]) AS v) AS ma
            FROM temporal.lmr_climate l
            JOIN gaz.clio_polities p ON p.id = {pid}
            WHERE ST_Within(l.geom, p.geom)
        ) sub
    """
    print(pd.read_sql(sql2, conn).to_string(index=False))

Tang Dynasty slices:
        name  fromyear  toyear  n_basins  actual_from  lmr_ok
Tang Dynasty       623     625       158          700   False
Tang Dynasty       626     626       404          700   False
Tang Dynasty       627     627       410          700   False
Tang Dynasty       628     629       422          700   False
Tang Dynasty       630     632       436          700   False
Tang Dynasty       633     637       754          700   False
Tang Dynasty       638     640       754          700   False
Tang Dynasty       641     646       773          700   False
Tang Dynasty       647     655       825          700   False
Tang Dynasty       656     660       897          700   False
Tang Dynasty       661     665      1075          700   False
Tang Dynasty       666     673       943          700   False
Tang Dynasty       674     681       790          700   False
Tang Dynasty       682     691       448          700   False
Tang Dynasty       692     704       542         

In [18]:
# Cell 11 — Band T inputs audit: confirm nothing else reads the year inputs we're hiding
from pathlib import Path
from scripts.shared import db_utils

ROOT = Path(db_utils.__file__).parent.parent.parent
tmpl = ROOT / 'app' / 'templates' / 'sandbox_v3.html'
src  = tmpl.read_text()

ids_of_interest = ['v3-polity-from-year', 'v3-polity-to-year', 'v3-polity-t-year-row']
for el_id in ids_of_interest:
    lines = [(i+1, l.strip()) for i, l in enumerate(src.splitlines()) if el_id in l]
    print(f'\n--- {el_id} ({len(lines)} occurrences) ---')
    for lineno, text in lines:
        print(f'  L{lineno}: {text[:120]}')


--- v3-polity-from-year (6 occurrences) ---
  L278: <input type="number" id="v3-polity-from-year" class="form-control form-control-sm"
  L454: const fy = document.getElementById('v3-polity-from-year').value;
  L1535: document.getElementById('v3-polity-from-year').value  = '';
  L1655: document.getElementById('v3-polity-from-year').value   = s.fromyear;
  L1692: const fy = document.getElementById('v3-polity-from-year').value;
  L2026: ? document.getElementById('v3-polity-from-year').value

--- v3-polity-to-year (6 occurrences) ---
  L281: <input type="number" id="v3-polity-to-year" class="form-control form-control-sm"
  L455: const ty = document.getElementById('v3-polity-to-year').value;
  L1536: document.getElementById('v3-polity-to-year').value    = '';
  L1656: document.getElementById('v3-polity-to-year').value     = s.toyear;
  L1693: const ty = document.getElementById('v3-polity-to-year').value;
  L2029: ? document.getElementById('v3-polity-to-year').value

--- v3-polity-t-year-ro